## Importing the Models

In [7]:
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor
from sklearn.metrics import mean_absolute_error,  mean_squared_error, r2_score
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split

## Train-Test Split

In [10]:
import pandas as pd

# 1. Load the data directly from Colab's default storage
print("Loading data...")
df = pd.read_csv('/clean_intern_data.csv')

# 2. Separate features and target
X = df.drop(columns=["Performance_Score"])
y = df["Performance_Score"]

# 3. 80/20 split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print(f"Training data ready: {X_train.shape}")

Loading data...
Training data ready: (7236, 4)


## Evaluating function

In [11]:
def evaluate(name, model, X_test, y_test):
    preds = model.predict(X_test)
    mae = mean_absolute_error(y_test, preds)
    mse = mean_squared_error(y_test, preds)
    rmse = np.sqrt(mean_squared_error(y_test, preds))
    r2 = r2_score(y_test, preds)
    print(f"\n── {name} ──")
    print(f"  MAE  : {mae:.3f}")
    print(f"  RMSE : {rmse:.3f}")
    print(f"  R²   : {r2:.3f}")
    return {"Model": name, "MAE": mae, "RMSE": rmse, "R2": r2}

## Training the Baseline Models

In [12]:
# 5. Train Random Forest Baseline
print("Training Random Forest...")
rf_base = RandomForestRegressor(random_state=42)
rf_base.fit(X_train, y_train)
rf_base_results = evaluate("Random Forest (Baseline)", rf_base, X_test, y_test)

# 6. Train XGBoost Baseline
print("Training XGBoost...")
xgb_base = XGBRegressor(random_state=42, verbosity=0)
xgb_base.fit(X_train, y_train)
xgb_base_results = evaluate("XGBoost (Baseline)", xgb_base, X_test, y_test)

Training Random Forest...

── Random Forest (Baseline) ──
  MAE  : 7.883
  RMSE : 9.918
  R²   : 0.566
Training XGBoost...

── XGBoost (Baseline) ──
  MAE  : 7.792
  RMSE : 9.835
  R²   : 0.574


## Hyperparameter Tuning

In [13]:
from sklearn.model_selection import RandomizedSearchCV

print("Initiating Hyperparameter Tuning...")

# 1. Random Forest Tuning
rf_param_grid = {
    "n_estimators"      : [100, 200, 300, 500],
    "max_depth"         : [None, 5, 10, 15, 20],
    "min_samples_split" : [2, 5, 10],
    "min_samples_leaf"  : [1, 2, 4],
    "max_features"      : ["sqrt", "log2", 1.0],
}

print("\nTuning Random Forest...")
rf_search = RandomizedSearchCV(
    RandomForestRegressor(random_state=42),
    param_distributions = rf_param_grid,
    n_iter      = 50,        # try 50 random combinations
    cv          = 5,         # 5-fold cross validation
    scoring     = "r2",
    random_state= 42,
    n_jobs      = -1,        # use all Colab CPU cores
    verbose     = 1
)
rf_search.fit(X_train, y_train)

print(f"Best RF R²     : {rf_search.best_score_:.3f}")
print(f"Best RF Params : {rf_search.best_params_}")


# 2. XGBoost Tuning
xgb_param_grid = {
    "n_estimators"  : [100, 200, 300, 500],
    "max_depth"     : [3, 4, 5, 6, 8],
    "learning_rate" : [0.01, 0.05, 0.1, 0.2],
    "subsample"     : [0.6, 0.8, 1.0],
    "colsample_bytree": [0.6, 0.8, 1.0],
    "reg_alpha"     : [0, 0.1, 0.5],       # L1 regularization
    "reg_lambda"    : [1, 1.5, 2],          # L2 regularization
}

print("\nTuning XGBoost...")
xgb_search = RandomizedSearchCV(
    XGBRegressor(random_state=42, verbosity=0),
    param_distributions = xgb_param_grid,
    n_iter      = 50,
    cv          = 5,
    scoring     = "r2",
    random_state= 42,
    n_jobs      = -1,
    verbose     = 1
)
xgb_search.fit(X_train, y_train)

print(f"Best XGB R²     : {xgb_search.best_score_:.3f}")
print(f"Best XGB Params : {xgb_search.best_params_}")

Initiating Hyperparameter Tuning...

Tuning Random Forest...
Fitting 5 folds for each of 50 candidates, totalling 250 fits
Best RF R²     : 0.598
Best RF Params : {'n_estimators': 200, 'min_samples_split': 5, 'min_samples_leaf': 4, 'max_features': 'sqrt', 'max_depth': 10}

Tuning XGBoost...
Fitting 5 folds for each of 50 candidates, totalling 250 fits
Best XGB R²     : 0.622
Best XGB Params : {'subsample': 0.8, 'reg_lambda': 2, 'reg_alpha': 0, 'n_estimators': 200, 'max_depth': 3, 'learning_rate': 0.05, 'colsample_bytree': 0.8}


## Extracting the Best Models

In [16]:
import pandas as pd
import joblib
import os

# RandomizedSearchCV already retrained the model on the full X_train using the best params
best_rf = rf_search.best_estimator_
best_xgb = xgb_search.best_estimator_

# Let's give them the final blind exam (X_test)
tuned_rf_results  = evaluate("Random Forest (Tuned)", best_rf,  X_test, y_test)
tuned_xgb_results = evaluate("XGBoost (Tuned)",       best_xgb, X_test, y_test)


# ── Step 5: Compare All Models ─────────────────────────────────
all_results = pd.DataFrame([
    rf_base_results,
    xgb_base_results,
    tuned_rf_results,
    tuned_xgb_results
])

print("\n Final Model Comparison ")
print(all_results.to_string(index=False))

# Create a reports folder and save the CSV
os.makedirs("reports", exist_ok=True)
all_results.to_csv("reports/model_comparison.csv", index=False)


# ── Step 6: Save the Final Models ──────────────────────────────
os.makedirs("models", exist_ok=True)
joblib.dump(best_rf,  "models/rf_model.pkl")
joblib.dump(best_xgb, "models/xgb_model.pkl")

print("\n Models serialized and saved to the /models/ folder!")


── Random Forest (Tuned) ──
  MAE  : 7.618
  RMSE : 9.513
  R²   : 0.601

── XGBoost (Tuned) ──
  MAE  : 7.415
  RMSE : 9.295
  R²   : 0.619

 Final Model Comparison 
                   Model      MAE     RMSE       R2
Random Forest (Baseline) 7.883247 9.917764 0.566441
      XGBoost (Baseline) 7.791629 9.835251 0.573625
   Random Forest (Tuned) 7.617566 9.512802 0.601124
         XGBoost (Tuned) 7.414576 9.294973 0.619183

 Models serialized and saved to the /models/ folder!
